# Task 2
Author: Laurent Lao
ID: 40020483

# Setup


In [1]:
import pandas as pd
data_source = "./Data_Sources"
clean_dir = "./Cleaned_Source"
quarantine_dir = "./Quarantined_Source"
current_date = "2026-11-18"


# Functions


In [2]:
# show row counts and optional previews
def debug_examine_table(tables, source_name, print_head=True):
  for table_name, df in tables.items():
    print(source_name, table_name, len(df))

  if print_head:
    for table_name, df in tables.items():
      print(df.head())

# check values using map and ordinary list membership
def matches_values(values, allowed):
    known_values = []
    for value in allowed:
        if pd.isna(value) == False:
            known_values.append(value)
    return values.map(lambda value: False if pd.isna(value) else value in known_values).astype(bool)


# count repeated values using groupby
def repeated_values(df, field):
    counts = df.groupby(field)[field].count()
    missing_count = df[field].isna().sum()
    return df[field].map(
        lambda value: missing_count > 1 if pd.isna(value) else counts.loc[value] > 1
    ).astype(bool)


# Source A


In [3]:
SourceA_Files = [
    "patients",
    "admissions",
    "clinical_notes",
    "diagnoses",
    "diagnostic_orders",
    "discharge_records",
    "icd9_dict",
    "icu_stays",
    "professionals",
    "radiology_examinations",
    "radiology_reports",
    "surgical_procedures",
    "triage_assessments"
]

A_data = {}

# load the pipe-separated files
for file_name in SourceA_Files:
    path = f"{data_source}/SourceA/SourceA_{file_name}.txt"
    # keep blank values and literal none when loading
    A_data[file_name] = pd.read_csv(path, sep="|", skiprows=1, dtype="string", keep_default_na=False)
    

# separate professionals into physicians and nurses
A_tables = A_data.copy()
professionals = A_tables.pop("professionals")
roles = professionals["role"].map(lambda value: value.strip().lower())
for role in ["physician", "nurse"]:
    A_tables[role] = professionals.loc[roles == role].copy()

debug_examine_table(A_tables, "A", False)

A patients 800
A admissions 1200
A clinical_notes 1113
A diagnoses 2119
A diagnostic_orders 275
A discharge_records 1176
A icd9_dict 20
A icu_stays 355
A radiology_examinations 275
A radiology_reports 260
A surgical_procedures 247
A triage_assessments 1200
A physician 15
A nurse 10


# Source B


In [4]:
SourceB_Files = [
    "admissions",
    "Clinical_Note",
    "diagnoses",
    "diagnostic_orders",
    "discharge_records",
    "ICU_Stay",
    "patients",
    "Radiology",
    "surgical_procedures",
    "triage_assessments"
]

B_data = {}

# load the csv files
for file_name in SourceB_Files:
    path = f"{data_source}/SourceB/SourceB_{file_name}.csv"
    # keep blank values and literal none when loading
    B_data[file_name] = pd.read_csv(path, dtype="string", keep_default_na=False)
    

B_tables = B_data.copy()

debug_examine_table(B_tables, "B", False)

B admissions 1200
B Clinical_Note 1053
B diagnoses 2133
B diagnostic_orders 307
B discharge_records 1148
B ICU_Stay 360
B patients 800
B Radiology 301
B surgical_procedures 237
B triage_assessments 1175


# Source C


In [5]:
# map tables to xml tags
SourceC_Tags = {
    "Patient": "Patient",
    "Admission": "Admission",
    "Triage": "Triage",
    "ICU_Stay": "Stay",
    "Surgical_Procedure": "Procedure",
    "Clinical_Note": "Note",
    "Diagnostic_Order": "Order",
    "Radiology": "Exam",
    "Discharge_Record": "Discharge",
}

C_data = {}

path = f"{data_source}/SourceC/SourceC_Hospital.xml"

for table_name, tag in SourceC_Tags.items():
    C_data[table_name] = pd.read_xml(path, xpath=f".//{tag}", dtype="string")
# handle diagnosis separately
C_data["Diagnosis"] = pd.read_xml(path, xpath=".//Diagnosis", dtype="string")

# need to flatten the xml
import xml.etree.ElementTree as ET
root = ET.parse(path).getroot()

# map output columns to nested xml fields
SourceC_Flatten_Fields = {
    "Patient": {
        "Address.Street": "Address/Street",
        "Address.City": "Address/City",
    },
    "Radiology": {
        "Report.Findings": "Report/Findings",
        "Report.Conclusion": "Report/Conclusion",
    },
}

# flatten the mapped fields
for table_name, fields in SourceC_Flatten_Fields.items():
    elements = root.findall(f".//{SourceC_Tags[table_name]}")
    if len(elements) != len(C_data[table_name]):
        raise ValueError(f"Source C {table_name}: XML row count does not match the loaded table.")
    nested_columns = []
    for output_column, xml_path in fields.items():
        C_data[table_name][output_column] = [element.findtext(xml_path) for element in elements]
        nested_column = xml_path.split("/")[0]
        if nested_column not in nested_columns:
            nested_columns.append(nested_column)
    C_data[table_name] = C_data[table_name].drop(nested_columns, axis=1)

C_data["Admission"] = C_data["Admission"].drop("Diagnoses", axis=1)

# link diagnosis to admissions
admission_references = []
for admission in root.findall(".//Admission"):
    for diagnosis in admission.findall("./Diagnoses/Diagnosis"):
        admission_references.append(admission.get("id"))
if len(admission_references) != len(C_data['Diagnosis']):
    raise ValueError("Source C diagnoses: admission reference count does not match the loaded rows.")
C_data["Diagnosis"]["admission_ref"] = admission_references

# combine first and lastname into one field
C_tables = C_data.copy()
C_tables["Patient"] = C_data["Patient"].copy()
C_tables["Patient"]["name"] = (
    C_tables["Patient"]["FirstName"].fillna("") + " "
    + C_tables["Patient"]["LastName"].fillna("")
).map(lambda value: value.strip())


# preserve original data before changes
C_original_tables = {}
for table_name, df in C_tables.items():
    C_original_tables[table_name] = df.copy()

# generate diagnosis ids
diagnosis_ids = []
for row in range(len(C_tables["Diagnosis"])):
    source_row = row + 1
    generated = source_row
    diagnosis_ids.append(generated)
C_tables["Diagnosis"]["diagnosis_id"] = diagnosis_ids

debug_examine_table(C_tables, "C", True)

C Patient 800
C Admission 1200
C Triage 1200
C ICU_Stay 358
C Surgical_Procedure 224
C Clinical_Note 1075
C Diagnostic_Order 290
C Radiology 290
C Discharge_Record 1167
C Diagnosis 2195
  id FirstName   LastName  ... Address.Street Address.City               name
0  1   Valeria  Rodriguez  ...       Calle 45    Barcelona  Valeria Rodriguez
1  2     Lucia   Gonzalez  ...       Calle 44       Madrid     Lucia Gonzalez
2  3     Mateo   Martinez  ...       Calle 62     Valencia     Mateo Martinez
3  4    Carmen   Martinez  ...       Calle 56       Madrid    Carmen Martinez
4  5      Hugo      Lopez  ...       Calle 32     Zaragoza         Hugo Lopez

[5 rows x 9 columns]
  id patient_ref  ...                   Reason DoctorId
0  1         219  ...           Dolor torácico        6
1  2         420  ...  Dificultad respiratoria        8
2  3           1  ...                   Fiebre        5
3  4          89  ...           Dolor torácico        6
4  5          38  ...  Dificultad respirator

# Source D


In [6]:
import json

# map table names to the current filenames
SourceD_Files = {
    "patients": "patients",
    "admissions": "admissions",
    "triage": "triage",
    "icu": "icu_stays",
    "surgery": "surgical_procedures",
    "notes": "clinical_notes",
    "orders": "diagnotic_orders",
    "discharge": "discharge_records",
}

D_data = {}
for table_name, file_name in SourceD_Files.items():
    path = f"{data_source}/SourceD/SourceD_{file_name}.json"
    D_data[table_name] = pd.read_json(path, dtype=False)

# map nested objects to the fields to flatten
SourceD_Flatten_Fields = {
    "patients": {
        "name": ["first", "last"],
        "contact": ["phone", "city"],
    },
    "triage": {
        "vitals": ["heartRate", "oxygenSaturation", "bloodPressure"],
    },
}

# move these fields into separate tables
SourceD_Child_Fields = {
    "admissions": ["diagnosis_codes", "primary_diagnosis", "diagnosis_timestamp"],
    "orders": ["examTimestamp", "technician", "report_findings", "report_conclusion"],
}
SourceD_Diagnosis_Fields = {
    "visitId": "visitId",
    "diagnosisTimestamp": "diagnosis_timestamp",
}
SourceD_Radiology_Fields = {
    "orderId": "orderId",
    "examTimestamp": "examTimestamp",
    "technician": "technician",
    "report.findings": "report_findings",
    "report.conclusion": "report_conclusion",
}

# keep these lists as json text
SourceD_JSON_Fields = {"triage": ["symptoms"]}


# flatten mapped objects and keep the other fields
def flatten_json_record(record, fields, child_fields=()):
    flattened = {}
    for column, value in record.items():
        if column not in fields and column not in child_fields:
            flattened[column] = value
    for nested_column, nested_fields in fields.items():
        nested = record[nested_column]
        if isinstance(nested, dict):
            for field in nested_fields:
                flattened[f"{nested_column}.{field}"] = nested[field]
        else:
            flattened[nested_column] = nested
    return flattened


# build the main tables
D_tables = {}
for table_name, df in D_data.items():
    fields = SourceD_Flatten_Fields.get(table_name, {})
    child_fields = SourceD_Child_Fields.get(table_name, [])
    records = []
    for row in range(len(df)):
        record = {}
        for column in df.columns:
            record[column] = df.loc[row, column]
        records.append(flatten_json_record(record, fields, child_fields))
    D_tables[table_name] = pd.DataFrame(records)

# split diagnosis codes into separate rows
records = []
df = D_data["admissions"]
for row in range(len(df)):
    codes = df.loc[row, "diagnosis_codes"]
    if pd.isna(codes):
        continue
    for code in str(codes).split(","):
        record = {}
        for output_field, source_field in SourceD_Diagnosis_Fields.items():
            record[output_field] = df.loc[row, source_field]
        record["icd9Code"] = code.strip()
        primary_code = df.loc[row, "primary_diagnosis"]
        record["isPrimary"] = pd.NA
        if pd.isna(primary_code) == False:
            record["isPrimary"] = code.strip() == str(primary_code).strip()
        records.append(record)
D_tables["diagnosis"] = pd.DataFrame(
    records, columns=["visitId", "diagnosisTimestamp", "icd9Code", "isPrimary"],
)

# take the examination fields from each order
records = []
df = D_data["orders"]
for row in range(len(df)):
    record = {}
    for output_field, source_field in SourceD_Radiology_Fields.items():
        record[output_field] = df.loc[row, source_field]
    records.append(record)
D_tables["radiology_examination"] = pd.DataFrame(records, columns=list(SourceD_Radiology_Fields))

# combine first and last names into one field
D_tables["patients"]["name"] = (
    D_tables["patients"]["name.first"].fillna("") + " "
    + D_tables["patients"]["name.last"].fillna("")
).map(lambda value: value.strip())

# convert the mapped lists to json text
for table_name, fields in SourceD_JSON_Fields.items():
    for field in fields:
        D_tables[table_name][field] = D_tables[table_name][field].map(json.dumps)

# keep original values
D_original_tables = {}
for table_name, df in D_tables.items():
    D_original_tables[table_name] = df.copy()

# generate ids
technical_id_fields = {
    "diagnosis": "diagnosis_id",
    "radiology_examination": "examination_id",
}
for table_name, key_field in technical_id_fields.items():
    generated_ids = []
    for row in range(len(D_tables[table_name])):
        generated = row + 1
        generated_ids.append(generated)
    D_tables[table_name][key_field] = generated_ids

debug_examine_table(D_tables, "D", True)


D patients 800
D admissions 1200
D triage 1200
D icu 351
D surgery 251
D notes 1067
D orders 307
D discharge 1136
D diagnosis 2155
D radiology_examination 307
   patientId  dobTimestamp  ... contact.city           name
0          1     616851666  ...        Tokyo     Haruto Ito
1          2    -465999533  ...        Kyoto  Sakura Suzuki
2          3    -122559533  ...        Tokyo     Yui Suzuki
3          4    -100009133  ...        Osaka       Aoi Sato
4          5    -267279533  ...        Osaka  Aoi Takahashi

[5 rows x 8 columns]
   visitId  patientId  ...  doctorId  dischargeTimestamp
0        1        219  ...         1        1.786846e+09
1        2        324  ...         6        1.768428e+09
2        3         26  ...        10        1.780286e+09
3        4         71  ...        13        1.776010e+09
4        5        728  ...         2        1.772762e+09

[5 rows x 7 columns]
   triageId  visitId  ...  vitals.oxygenSaturation vitals.bloodPressure
0         1        1  .

# Name mapping


In [7]:
# map source names to common names
Name_Mappings = {
    'patient': ['patient', 'patients'],
    'admission': ['admission', 'admissions'],
    'clinical_note': ['clinical_note', 'clinical_notes', 'notes', 'note'],
    'diagnosis': ['diagnosis', 'diagnoses'],
    'diagnostic_order': ['diagnostic_order', 'diagnostic_orders', 'orders', 'order'],
    'discharge_record': ['discharge_record', 'discharge_records', 'discharge'],
    'icd9_dict': ['icd9_dict'],
    'icu_stay': ['icu_stay', 'icu_stays', 'icu', 'stay'],
    'radiology_examination': ['radiology_examination', 'radiology_examinations', 'radiology', 'exam'],
    'radiology_report': ['radiology_report', 'radiology_reports'],
    'surgical_procedure': ['surgical_procedure', 'surgical_procedures', 'surgery', 'procedure'],
    'triage_assessment': ['triage_assessment', 'triage_assessments', 'triage'],
    'physician': ['physician'],
    'nurse': ['nurse'],
    'physician_id': ['physician_id', 'author_mrn', 'authorid'],
    'name': ['name', 'full_name'],
    'nurse_id': ['nurse_id', 'nurse_mrn', 'nurseid'],
    'patient_id': ['patient_id', 'mrn', 'patientid', 'patient_mrn', 'patient_ref'],
    'health_insurance_number': ['health_insurance_number', 'insurance_id'],
    'date_birth': ['date_birth', 'dob', 'dateofbirth', 'dobtimestamp'],
    'gender': ['gender', 'gender_cd'],
    'phone_number': ['phone_number', 'contact_phone', 'phone', 'contact.phone'],
    'address': ['address', 'home_address', 'address.street'],
    'admission_id': ['admission_id', 'visit_id', 'visitid', 'admission_ref'],
    'admitting_professional_id': ['admitting_professional_id', 'admitting_doctor'],
    'admission_date_time': ['admission_date_time', 'admit_date', 'admissiondate', 'admittimestamp'],
    'discharge_date_time': ['discharge_date_time', 'discharge_date', 'dischargedate', 'dischargetimestamp', 'discharge_datetime'],
    'reason_admission': ['reason_admission', 'reason_text', 'reason'],
    'type': ['type', 'note_format'],
    'creation_date': ['creation_date', 'created_datetime', 'notetimestamp'],
    'diagnosis_id': ['diagnosis_id', 'diag_id'],
    'diagnosis_date_time': ['diagnosis_date_time', 'diag_datetime', 'diagnosistimestamp'],
    'primary_diagnosis': ['primary_diagnosis', 'is_primary', 'isprimary'],
    'diagnosing_physician_id': ['diagnosing_physician_id', 'attending_doctor'],
    'requesting_physician_id': ['requesting_physician_id'],
    'requested_examination': ['requested_examination', 'exam_type', 'examtype'],
    'request_date_time': ['request_date_time', 'order_datetime', 'orderdate', 'ordertimestamp'],
    'priority': ['priority', 'priority_level'],
    'discharge_record_id': ['discharge_record_id', 'discharge_id', 'dischargeid'],
    'discharge_destination': ['discharge_destination', 'disposition_code', 'destination'],
    'responsible_physician': ['responsible_physician', 'discharging_doctor_mrn'],
    'followup_appointments': ['followup_appointments', 'followup_date'],
    'unit_stay_id': ['unit_stay_id', 'stay_id', 'stayid'],
    'responsible_physician_id': ['responsible_physician_id'],
    'unit_type': ['unit_type', 'ward_type', 'wardtype'],
    'entry_date_time': ['entry_date_time', 'in_date', 'entrydate', 'entrytimestamp'],
    'exit_date_time': ['exit_date_time', 'out_date', 'exitdate', 'exittimestamp'],
    'reason_transfer': ['reason_transfer', 'transfer_reason'],
    'examination_id': ['examination_id', 'exam_id'],
    'radiology_technician': ['radiology_technician', 'technician_name', 'technician'],
    'examination_date_time': ['examination_date_time', 'scan_datetime', 'scandate', 'examtimestamp'],
    'procedure_id': ['procedure_id', 'surgery_id', 'surgeryid'],
    'surgeon_id': ['surgeon_id', 'surgeon_mrn', 'surgeonid'],
    'procedure_name': ['procedure_name', 'proc_name', 'procedurename'],
    'procedure_code': ['procedure_code', 'proc_code', 'procedurecode'],
    'procedure_date_time': ['procedure_date_time', 'surgery_datetime', 'surgerytimestamp'],
    'heart_rate': ['heart_rate', 'hr_bpm', 'heartrate', 'vitals.heartrate'],
    'oxygen_saturation': ['oxygen_saturation', 'o2_sat', 'oxygensaturation', 'vitals.oxygensaturation'],
    'triage_priority_level': ['triage_priority_level', 'esi_priority', 'triagepriority'],
    'known_allergies': ['known_allergies', 'allergy_note'],
    'city': ['city', 'address.city', 'contact.city'],
    'admission_type': ['admission_type', 'admissiontype'],
    'triage_id': ['triage_id', 'triageid'],
    'icd9_code': ['icd9_code', 'icd9code'],
    'note_id': ['note_id', 'noteid'],
    'note_text': ['note_text', 'body', 'notetext'],
    'order_id': ['order_id', 'orderid', 'order_ref'],
    'condition_at_discharge': ['condition_at_discharge', 'condition'],
    'blood_pressure': ['blood_pressure', 'vitals.bloodpressure'],
    'first_name': ['first_name', 'firstname', 'name.first'],
    'last_name': ['last_name', 'lastname', 'name.last'],
}

# some column names depend on the table
Ambiguous_Columns = {
    'physician': {'professional_id': 'physician_id'},
    'nurse': {'professional_id': 'nurse_id'},
    'patient': {'id': 'patient_id'},
    'admission': {'id': 'admission_id', 'doctorid': 'admitting_professional_id'},
    'clinical_note': {'doctorid': 'physician_id', 'id': 'note_id'},
    'diagnosis': {'date': 'diagnosis_date_time', 'doctorid': 'diagnosing_physician_id'},
    'diagnostic_order': {'doctor_mrn': 'requesting_physician_id', 'doctorid': 'requesting_physician_id', 'id': 'order_id'},
    'discharge_record': {'id': 'discharge_record_id'},
    'icu_stay': {'id': 'unit_stay_id', 'doctor_mrn': 'responsible_physician_id', 'doctorid': 'responsible_physician_id'},
    'radiology_examination': {'id': 'examination_id'},
    'surgical_procedure': {'id': 'procedure_id', 'name': 'procedure_name', 'date': 'procedure_date_time'},
    'triage_assessment': {'priority': 'triage_priority_level', 'id': 'triage_id'},
}

# Rename columns


In [8]:
def standardize_name(name):
    # trim spaces and lowercase names for matching
    name = name.strip().lower()
    # rename to matching
    for standard_name, variations in Name_Mappings.items():
        if name in variations:
            return standard_name
    return name

def standardize_source(data):
    normalized = {}
    for table_name, df in data.items():
        table_name = standardize_name(table_name)
        column_names = {}
        for column in df.columns:
            name = column.strip().lower()
            # check for ambiguous columns first
            name = Ambiguous_Columns.get(table_name, {}).get(name, name)
            column_names[column] = standardize_name(name)
        normalized[table_name] = df.rename(columns=column_names)
    return normalized

# Field mapping


In [9]:
# required fields from the schema
Required_Fields = {
    "icd9_dict": ["icd9_code", "official_diagnosis_title"],
    "physician": ["physician_id"],
    "nurse": ["nurse_id", "name"],
    "patient": ["patient_id", "name"],
    "admission": ["admission_id", "admission_date_time"],
    "triage_assessment": ["triage_id"],
    "diagnosis": ["diagnosis_id", "diagnosis_date_time"],
    "icu_stay": ["unit_stay_id", "unit_type", "entry_date_time"],
    "clinical_note": ["note_id", "creation_date"],
    "diagnostic_order": ["order_id"],
    "radiology_examination": ["examination_id"],
    "radiology_report": ["radiology_report_id"],
    "discharge_record": ["discharge_record_id", "discharge_date_time", "discharge_status"],
    "surgical_procedure": ["procedure_id", "procedure_name", "procedure_date_time"],
}

# values to treat as missing
Missing_Markers = [
    "", "N/A", "n/a", "NULL", "null", "Unknown", "unknown", "UNKNOWN", "UNK", "unk",
]

# map procedure codes to names
Procedure_Names = {
    "PROC-APP": "Appendectomy",
    "PROC-CHOL": "Cholecystectomy",
    "PROC-CABG": "Coronary Artery Bypass",
    "PROC-BOWEL": "Bowel Resection",
    "PROC-KNEE": "Knee Replacement",
}

# map category values to common labels
Category_Maps = {
    "admission_type": {
        "EMERGENCY": "EMERGENCY", "URGENT": "URGENT",
        "ELECTIVE": "ELECTIVE", "OBSERVATION": "OBSERVATION",
    },
    "triage_priority_level": {
        "STAT": "STAT", "HIGH-PRIORITY EMERGENCY": "HIGH_PRIORITY_EMERGENCY",
        "URGENT": "URGENT", "ROUTINE": "ROUTINE",
        "ESI-1": "ESI-1", "ESI-2": "ESI-2", "ESI-3": "ESI-3",
        "ESI-4": "ESI-4", "ESI-5": "ESI-5",
    },
    "unit_type": {
        "ICU": "ICU", "MICU": "MICU", "SICU": "SICU", "CCU": "CCU",
        "WARD-ICU": "ICU", "WARD-SICU": "SICU", "WARD-CCU": "CCU",
    },
    "priority": {
        "STAT": "STAT", "URGENT": "URGENT", "ROUTINE": "ROUTINE",
        "1": "LEVEL_1", "2": "LEVEL_2", "3": "LEVEL_3", "4": "LEVEL_4",
    },
    "order_status": {"PENDING": "PENDING", "COMPLETED": "COMPLETED", "CANCELLED": "CANCELLED"},
    "examination_status": {
        "PENDING": "PENDING", "IN PROGRESS": "IN_PROGRESS", "IN_PROGRESS": "IN_PROGRESS",
        "FINALIZED": "FINALIZED", "COMPLETED": "COMPLETED", "CANCELLED": "CANCELLED",
    },
    "discharge_status": {"DISCHARGED": "DISCHARGED", "DECEASED": "DECEASED"},
    "gender": {"M": "MALE", "MALE": "MALE", "F": "FEMALE", "FEMALE": "FEMALE"},
    "role": {"PHYSICIAN": "PHYSICIAN", "NURSE": "NURSE"},
    "life_status": {"ALIVE": "ALIVE", "DECEASED": "DECEASED"},
    "current_condition": {
        "STABLE": "STABLE", "IMPROVING": "IMPROVING",
        "CRITICAL": "CRITICAL", "OBSERVATION": "OBSERVATION",
    },
    "condition_at_discharge": {
        "STABLE": "STABLE", "IMPROVED": "IMPROVED", "CRITICAL": "CRITICAL",
        "FULLY RECOVERED": "FULLY_RECOVERED",
    },
    "discharge_destination": {
        "HOME": "HOME", "SNF": "SKILLED_NURSING", "REHAB": "REHAB",
        "REHAB FACILITY": "REHAB", "LONG-TERM CARE": "LONG_TERM_CARE",
        "TRANSFER": "TRANSFER", "TRANSFER TO ANOTHER HOSPITAL": "TRANSFER",
        "MORTUARY": "MORTUARY", "EXP": "SOURCE_B_EXP",
    },
    "type": {
        "ADMISSION NOTE": "ADMISSION_NOTE", "PROGRESS NOTE": "PROGRESS_NOTE",
        "CONSULTATION NOTE": "CONSULTATION_NOTE", "SOAP": "SOAP",
        "NARRATIVE": "NARRATIVE", "DISCHARGE SUMMARY": "DISCHARGE_SUMMARY",
    },
    "procedure_name": {
        "APPENDECTOMY": "APPENDECTOMY", "CHOLECYSTECTOMY": "CHOLECYSTECTOMY",
        "BOWEL RESECTION": "BOWEL_RESECTION", "KNEE REPLACEMENT": "KNEE_REPLACEMENT",
        "CABG": "CORONARY_ARTERY_BYPASS_GRAFT",
        "CORONARY ARTERY BYPASS": "CORONARY_ARTERY_BYPASS_GRAFT",
        "CORONARY ARTERY BYPASS GRAFT": "CORONARY_ARTERY_BYPASS_GRAFT",
    },
}
# map boolean values
Boolean_Map = {"TRUE": True, "FALSE": False, "YES": True, "NO": False, "Y": True, "N": False}

# keep the priority scheme used by each source
Triage_Schemes = {"A": "LOCAL_A", "B": "ESI", "C": "ESI", "D": "ESI"}
Order_Priority_Schemes = {"A": "LOCAL_A", "B": "SOURCE_B_NUMERIC"}

# date fields and source formats
Date_Fields = [
    "date_birth", "admission_date_time", "discharge_date_time", "creation_date",
    "diagnosis_date_time", "request_date_time", "entry_date_time", "exit_date_time",
    "examination_date_time", "procedure_date_time", "followup_appointments",
]
Date_Only_Fields = ["date_birth", "followup_appointments"]
Event_Formats = {"A": "%Y-%m-%d %H:%M:%S", "B": "%d/%m/%Y %H:%M", "C": "%Y-%m-%dT%H:%M:%S"}


# prepare category text for matching
def category_text(value):
    if pd.isna(value):
        return pd.NA
    return " ".join(str(value).split()).upper()

# Standardize sources


In [10]:
# group tables by source
Source_Dict = {"A": A_tables, "B": B_tables, "C": C_tables, "D": D_tables}
Original_Source_Dict = {"A": A_tables, "B": B_tables, "C": C_original_tables, "D": D_original_tables}
Original_Dict = {}
Standardized_Dict = {}
Primary_Keys = {}
for table_name, fields in Required_Fields.items():
    Primary_Keys[table_name] = fields[0]

# some exceptions
Generated_Entities = {
    ("C", "diagnosis"): "diagnosis_id",
    ("D", "diagnosis"): "diagnosis_id",
    ("D", "radiology_examination"): "examination_id",
}

# standardize
for source_name, tables in Source_Dict.items():
    named_tables = standardize_source(tables)
    named_originals = standardize_source(Original_Source_Dict[source_name])
    Original_Dict[source_name] = {}
    Standardized_Dict[source_name] = {}
    for table_name, df in named_tables.items():
        df["_source"] = source_name
        df["_table"] = table_name
        df["_source_row"] = list(range(1, len(df) + 1))
        supplied = named_originals[table_name]
        supplied["_source"] = source_name
        supplied["_table"] = table_name
        supplied["_source_row"] = list(range(1, len(supplied) + 1))
        Original_Dict[source_name][table_name] = supplied
        Standardized_Dict[source_name][table_name] = df

# verification
pairs = Original_Dict["B"]["surgical_procedure"][["procedure_code", "procedure_name"]].drop_duplicates()
for row in pairs.index:
    procedure_code = pairs.loc[row, "procedure_code"]
    if Procedure_Names[procedure_code] != pairs.loc[row, 'procedure_name']:
        raise ValueError(f"Source B procedure {procedure_code}: procedure name does not match the mapping.")


# Standardize values


In [11]:
# apply category mappings and fill names from known codes
for source_name, tables in Standardized_Dict.items():
    for table_name, df in tables.items():
        if table_name == "surgical_procedure" and "procedure_code" in df:
            if "procedure_name" not in df:
                df["procedure_name"] = pd.NA
            missing_name = df["procedure_name"].isna()
            df.loc[missing_name, "procedure_name"] = df.loc[missing_name, "procedure_code"].map(Procedure_Names)

        for field, mapping in Category_Maps.items():
            if field in df:
                normalized = df[field].map(category_text)
                df[field] = normalized.astype(object).replace(mapping).astype(object)

        for field in ["primary_diagnosis", "secondary_diagnosis", "ventilator_used", "contrast_used"]:
            if field in df:
                normalized = df[field].map(category_text)
                df[field] = normalized.astype(object).replace(Boolean_Map).astype(object)

        if table_name == "admission" and "admission_type" not in df:
            df["admission_type"] = pd.NA
        if table_name == "triage_assessment":
            df["triage_scheme"] = Triage_Schemes[source_name]
        if table_name == "diagnostic_order" and "priority" in df:
            df["priority_scheme"] = Order_Priority_Schemes[source_name]

# check that standardization kept the same row counts
normalization_rows = []
for source_name, tables in Standardized_Dict.items():
    before = sum(len(df) for df in Source_Dict[source_name].values())
    after = sum(len(df) for df in tables.values())
    if before != after:
        raise ValueError(f"Source {source_name}: standardization changed the row count.")
    normalization_rows.append({"source": source_name, "before": before, "after": after})
Normalization_Summary = pd.DataFrame(normalization_rows)


# Cleaning setup


In [12]:
from os import makedirs

clean_output = clean_dir
quarantine_output = quarantine_dir
# copy the tables before cleaning
Cleaned_Dict = {}
for source_name, tables in Standardized_Dict.items():
    Cleaned_Dict[source_name] = {}
    for table_name, df in tables.items():
        Cleaned_Dict[source_name][table_name] = df.copy()

Quarantine_Parts = []

# keep original values for removed rows
def save_removed(source, table_name, before, after, reason):
    removed = before[matches_values(before["_source_row"], after["_source_row"]) == False]
    if len(removed) > 0:
        original = Original_Dict[source][table_name]
        rejected = original[matches_values(original["_source_row"], removed["_source_row"])].copy()
        rejected["_reason"] = reason
        rejected["_missing_required_fields"] = ""
        if "_missing_required_fields" in removed:
            # match missing fields using the original index
            rejected["_missing_required_fields"] = removed["_missing_required_fields"]
        Quarantine_Parts.append(rejected)
    return after

# remove failed rows and save them in quarantine
def reject_rows(source, table_name, df, bad, reason):
    bad = bad.fillna(False)
    kept = df[bad == False].copy()
    return save_removed(source, table_name, df, kept, reason)


# Missing data


In [13]:
# remove rows with missing required values


def trim_text(value):
    if isinstance(value, str):
        return value.strip()
    return value


for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        required = Required_Fields[table_name]
        for field in required:
            if field not in df:
                df[field] = pd.NA

        for field in df.columns:
            if field.startswith("_") == False:
                df[field] = df[field].map(trim_text)
        df = df.replace(Missing_Markers, pd.NA)

        # record which required fields are missing
        df["_missing_required_fields"] = ""
        for field in required:
            missing = df[field].isna()
            df.loc[missing, "_missing_required_fields"] = df.loc[missing, "_missing_required_fields"] + field + "; "
        kept = df.dropna(subset=required).copy()
        df = save_removed(source, table_name, df, kept, "missing_required_information")
        df = df.drop("_missing_required_fields", axis=1)
        tables[table_name] = df


# Duplicates


In [14]:
# remove repeated rows and conflicting ids
def check_duplicates(source, table_name, df):
    # leave generated ids out of the row comparison
    generated_field = Generated_Entities.get((source, table_name))
    supplied_columns = []
    for column in df.columns:
        if column.startswith("_") == False and column != generated_field:
            supplied_columns.append(column)

    kept = df.drop_duplicates(subset=supplied_columns, keep="first").copy()
    df = save_removed(source, table_name, df, kept, "exact_or_normalized_duplicate")

    key = Primary_Keys[table_name]
    bad = repeated_values(df, key)
    df = reject_rows(source, table_name, df, bad, "conflicting_entity_key:" + key)
    if table_name == "patient" and "health_insurance_number" in df:
        bad = (df["health_insurance_number"].isna() == False) & repeated_values(df, "health_insurance_number")
        df = reject_rows(source, table_name, df, bad, "possible_duplicate_patient:insurance_number")
    return df


for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        df = check_duplicates(source, table_name, df)
        tables[table_name] = df



# Text


In [15]:
# clean spaces in short text fields and keep note formatting
Short_Text_Fields = [
    "name", "first_name", "last_name", "address", "city", "specialty",
    "procedure_name", "requested_examination", "examination_type",
    "body_site", "body_area_examined", "radiology_technician",
]

def collapse_spaces(value):
    if isinstance(value, str):
        return " ".join(value.split())
    return value

for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        for column in Short_Text_Fields:
            if column in df:
                df[column] = df[column].map(collapse_spaces)
        df = check_duplicates(source, table_name, df)
        tables[table_name] = df


# Date_Time formats


In [16]:
# parse dates using the source format
for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        df["_time_basis"] = "LOCAL_ZONE_UNKNOWN"
        if source == "D":
            df["_time_basis"] = "UTC"
        for column in Date_Fields:
            if column not in df:
                continue
            raw = df[column].copy()
            if source == "D":
                seconds = pd.to_numeric(raw, errors="coerce")
                parsed = pd.to_datetime(seconds, unit="s", utc=True, errors="coerce")
                invalid = (raw.isna() == False) & (parsed.isna() | (seconds % 1 != 0))
            else:
                date_format = Event_Formats[source]
                if column in Date_Only_Fields:
                    date_format = "%Y-%m-%d"
                    if source == "B":
                        date_format = "%d/%m/%Y"
                    if source == "C" and column == "date_birth":
                        date_format = Event_Formats["C"]
                text = raw.astype("string")
                parsed = pd.to_datetime(text, format=date_format, errors="coerce")
                invalid = (raw.isna() == False) & (parsed.isna() | (parsed.dt.strftime(date_format) != text))
            df = reject_rows(source, table_name, df, invalid, "invalid_datetime:" + column)
            df[column] = parsed.loc[df.index]
            if column in Date_Only_Fields:
                # keep the date without the time
                df[column] = df[column].dt.normalize()

        # remove birth dates after the review date
        if table_name == "patient":
            cutoff = pd.to_datetime(current_date)
            if source == "D":
                cutoff = pd.to_datetime(current_date, utc=True)
            df = reject_rows(source, table_name, df, df["date_birth"] > cutoff, "birth_date_after_review_date")
        tables[table_name] = df


# Numeric values


In [17]:
MRN_Fields = [
    "patient_id", "nurse_id", "physician_id", "admitting_professional_id",
    "diagnosing_physician_id", "requesting_physician_id", "responsible_physician_id",
    "responsible_physician", "surgeon_id", "referring_physician_id",
]
# lower bound, upper bound and whether the lower bound is included
Numeric_Bounds = {
    "heart_rate": [0, 300, False],
    "respiratory_rate": [0, 100, False],
    "oxygen_saturation": [0, 100, False],
    "body_temperature": [25, 45, True],
    "pain_level": [0, 10, True],
    "bp_systolic": [0, 300, False],
    "bp_diastolic": [0, 300, False],
}


# check the mrn format
def valid_mrn(value):
    if pd.isna(value):
        return False
    text = str(value)
    return text.startswith("MRN-") and text[4:].isdigit() and int(text[4:]) > 0


# split blood pressure into two values
def pressure_part(value, position):
    if pd.isna(value):
        return pd.NA
    parts = str(value).split("/")
    if len(parts) != 2:
        return pd.NA
    return parts[position].strip()


# read scores written out of ten
def pain_number(value):
    if isinstance(value, str) and "/" in value:
        parts = value.split("/")
        if len(parts) == 2 and parts[1].strip() == "10":
            return parts[0].strip()
    return value

for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        # check supplied ids
        for column in df.columns:
            if column.startswith("_"):
                continue
            if column.endswith("_id") or column == "responsible_physician":
                if column == Generated_Entities.get((source, table_name)):
                    continue
                raw = df[column].copy()
                if source == "B" and column in MRN_Fields:
                    bad = (raw.isna() == False) & (raw.map(valid_mrn) == False)
                    df = reject_rows(source, table_name, df, bad, "invalid_mrn:" + column)
                    df[column] = df[column].astype("string")
                else:
                    number = pd.to_numeric(raw, errors="coerce")
                    bad = (raw.isna() == False) & (number.isna() | (number <= 0) | (number % 1 != 0) | (number > 9007199254740991))
                    df = reject_rows(source, table_name, df, bad, "invalid_positive_integer_id:" + column)
                    df[column] = number.loc[df.index].map(lambda value: value if pd.isna(value) else str(int(value))).astype("string")

        if "pain_level" in df:
            df["pain_level"] = df["pain_level"].map(pain_number)
        # convert fahrenheit to celsius
        if "temp_fahrenheit" in df:
            raw = df["temp_fahrenheit"].copy()
            number = pd.to_numeric(raw, errors="coerce")
            bad = (raw.isna() == False) & number.isna()
            df = reject_rows(source, table_name, df, bad, "invalid_numeric:temp_fahrenheit")
            df["temp_fahrenheit"] = number.loc[df.index]
            df["body_temperature"] = (df["temp_fahrenheit"] - 32) * 5 / 9
        # split and parse blood pressure
        if "blood_pressure" in df:
            raw = df["blood_pressure"].copy()
            systolic = raw.map(lambda value: pressure_part(value, 0))
            diastolic = raw.map(lambda value: pressure_part(value, 1))
            df["bp_systolic"] = pd.to_numeric(systolic, errors="coerce")
            df["bp_diastolic"] = pd.to_numeric(diastolic, errors="coerce")
            bad = (raw.isna() == False) & (df["bp_systolic"].isna() | df["bp_diastolic"].isna())
            df = reject_rows(source, table_name, df, bad, "invalid_blood_pressure_format")

        # remove values outside the review limits
        for column, bounds in Numeric_Bounds.items():
            if column not in df:
                continue
            raw = df[column].copy()
            number = pd.to_numeric(raw, errors="coerce")
            low, high, inclusive_low = bounds
            below = number <= low
            if inclusive_low:
                below = number < low
            bad = (raw.isna() == False) & (number.isna() | below | (number > high))
            df = reject_rows(source, table_name, df, bad, "numeric_value_requires_review:" + column)
            df[column] = number.loc[df.index]
        if "bp_systolic" in df and "bp_diastolic" in df:
            bad = (df["bp_systolic"].isna() == False) & (df["bp_diastolic"].isna() == False) & (df["bp_systolic"] <= df["bp_diastolic"])
            df = reject_rows(source, table_name, df, bad, "systolic_not_above_diastolic")

        df = check_duplicates(source, table_name, df)
        tables[table_name] = df

print("temperature: celsius, blood pressure: mmhg, oxygen saturation: percent")


temperature: celsius, blood pressure: mmhg, oxygen saturation: percent


# Categories


In [18]:
# check category and boolean values
for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        for field, mapping in Category_Maps.items():
            if field in df:
                allowed = list(mapping.values())
                bad = (df[field].isna() == False) & (matches_values(df[field], allowed) == False)
                df = reject_rows(source, table_name, df, bad, "unmapped_category:" + field)
                df[field] = df[field].astype("string")
        for field in ["primary_diagnosis", "secondary_diagnosis", "ventilator_used", "contrast_used"]:
            if field in df:
                bad = (df[field].isna() == False) & (matches_values(df[field], [True, False]) == False)
                df = reject_rows(source, table_name, df, bad, "invalid_boolean:" + field)
                df[field] = df[field].astype(object)
        if "primary_diagnosis" in df and "secondary_diagnosis" in df:
            bad = (df["primary_diagnosis"] == True) & (df["secondary_diagnosis"] == True)
            df = reject_rows(source, table_name, df, bad, "both_primary_and_secondary_diagnosis")
        df = check_duplicates(source, table_name, df)
        tables[table_name] = df


# References


In [19]:
# keep a copy for date checks
Before_Relationships = {}
for source, tables in Cleaned_Dict.items():
    Before_Relationships[source] = {}
    for table_name, df in tables.items():
        Before_Relationships[source][table_name] = df.copy()

# check parent tables before their children
Table_Order = [
    "icd9_dict", "physician", "nurse", "patient", "admission",
    "triage_assessment", "clinical_note", "diagnosis", "diagnostic_order",
    "icu_stay", "surgical_procedure", "discharge_record",
    "radiology_examination", "radiology_report",
]
# map child fields to parent tables
Parent_Links = {
    "admission": ["patient_id", "patient"],
    "triage_assessment": ["admission_id", "admission"],
    "clinical_note": ["admission_id", "admission"],
    "diagnosis": ["admission_id", "admission"],
    "diagnostic_order": ["admission_id", "admission"],
    "icu_stay": ["admission_id", "admission"],
    "surgical_procedure": ["admission_id", "admission"],
    "discharge_record": ["admission_id", "admission"],
    "radiology_examination": ["order_id", "diagnostic_order"],
}
# map staff references to their tables
Provider_Links = {
    "admitting_professional_id": "physician", "physician_id": "physician",
    "diagnosing_physician_id": "physician", "requesting_physician_id": "physician",
    "responsible_physician_id": "physician", "responsible_physician": "physician",
    "surgeon_id": "physician", "referring_physician_id": "physician",
    "nurse_id": "nurse",
}


def check_relationships():
    for source, tables in Cleaned_Dict.items():
        for table_name in Table_Order:
            if table_name not in tables:
                continue
            df = tables[table_name]
            if table_name in Parent_Links:
                field, parent_table = Parent_Links[table_name]
                if field not in df:
                    df[field] = pd.NA
                parent_key = Primary_Keys[parent_table]
                parent = tables[parent_table]
                bad = df[field].isna() | (matches_values(df[field], parent[parent_key]) == False)
                df = reject_rows(source, table_name, df, bad, "missing_or_quarantined_parent:" + parent_table)

            if table_name != "admission" and "admission_id" in df and "patient_id" in df:
                parent = tables["admission"][["admission_id", "patient_id"]].rename(columns={"patient_id": "_parent_patient"})
                compared = pd.merge(df, parent, on="admission_id", how="left")
                bad = (compared["patient_id"].isna() == False) & (compared["_parent_patient"].isna() == False) & (compared["patient_id"] != compared["_parent_patient"])
                bad_rows = compared.loc[bad, "_source_row"]
                df = reject_rows(source, table_name, df, matches_values(df["_source_row"], bad_rows), "patient_does_not_match_admission")

            if table_name not in ["physician", "nurse"]:
                for field, parent_table in Provider_Links.items():
                    if field not in df:
                        continue
                    if parent_table in tables:
                        parent_key = Primary_Keys[parent_table]
                        bad = (df[field].isna() == False) & (matches_values(df[field], tables[parent_table][parent_key]) == False)
                        reason = "missing_or_wrong_role_provider:" + field
                        df = reject_rows(source, table_name, df, bad, reason)

            if source == "A" and table_name == "diagnosis":
                dictionary = tables["icd9_dict"]["icd9_code"]
                bad = df["icd9_code"].isna() | (matches_values(df["icd9_code"], dictionary) == False)
                df = reject_rows(source, table_name, df, bad, "icd9_code_not_in_supplied_dictionary")
            if table_name == "radiology_report":
                # no examination or order reference is supplied
                df = reject_rows(source, table_name, df, (df["_source_row"].isna() == False), "report_has_no_examination_reference")

            tables[table_name] = df


# only source a supplies staff lists to check these ids
print("no staff lists for b, c or d; staff ids not checked")
check_relationships()


no staff lists for b, c or d; staff ids not checked


# Date_Time checks


In [20]:
# remove rows with invalid event dates
def date_time_rule(source, table_name, compared, bad, reason):
    bad_rows = compared.loc[bad.fillna(False), "_source_row"]
    df = Cleaned_Dict[source][table_name]
    Cleaned_Dict[source][table_name] = reject_rows(
        source, table_name, df, matches_values(df["_source_row"], bad_rows), reason,
    )


Event_Fields = [
    "creation_date", "diagnosis_date_time", "request_date_time",
    "procedure_date_time", "entry_date_time", "exit_date_time", "discharge_date_time",
]

for source, candidates in Before_Relationships.items():
    admissions = candidates["admission"]
    patients = candidates["patient"][["patient_id", "date_birth"]]
    compared = pd.merge(admissions, patients, on="patient_id", how="left")
    date_time_rule(source, 'admission', compared, compared['discharge_date_time'] < compared['admission_date_time'], 'discharge_before_admission')
    date_time_rule(source, 'admission', compared, compared['admission_date_time'] < compared['date_birth'], 'admission_before_birth')

    parent_times = admissions[["admission_id", "admission_date_time", "discharge_date_time"]].rename(
        columns={"discharge_date_time": "_parent_discharge"},
    )
    for table_name, df in candidates.items():
        if table_name == "admission" or "admission_id" not in df:
            continue
        compared = pd.merge(df, parent_times, on="admission_id", how="left")
        for field in Event_Fields:
            if field not in compared:
                continue
            date_time_rule(source, table_name, compared, compared[field] < compared['admission_date_time'], field + ':before_admission')
            date_time_rule(source, table_name, compared, compared[field] > compared['_parent_discharge'], field + ':after_discharge')

        if table_name == "discharge_record":
            bad = (compared["discharge_date_time"].isna() == False) & (compared["_parent_discharge"].isna() == False) & (compared["discharge_date_time"] != compared["_parent_discharge"])
            date_time_rule(source, table_name, compared, bad, 'discharge_record_disagrees_with_admission')
            bad = (compared["admission_date_time"].isna() == False) & compared["_parent_discharge"].isna()
            date_time_rule(source, table_name, compared, bad, 'discharge_record_but_admission_discharge_unknown')
            if "followup_appointments" in compared:
                bad = compared["followup_appointments"] < compared["discharge_date_time"].dt.normalize()
                date_time_rule(source, table_name, compared, bad, 'followup_before_discharge_date')

        if table_name == "icu_stay":
            date_time_rule(source, table_name, compared, compared['exit_date_time'] < compared['entry_date_time'], 'unit_exit_before_entry')
            bad = compared["exit_date_time"].isna() & (compared["_parent_discharge"].isna() == False)
            date_time_rule(source, table_name, compared, bad, 'open_unit_stay_in_completed_admission')

            # check for overlapping stays in the same admission
            overlap_rows = []
            for admission_id, group in df.groupby("admission_id"):
                group = group.sort_values("entry_date_time")
                for left in range(len(group)):
                    first = group.iloc[left]
                    for right in range(left + 1, len(group)):
                        second = group.iloc[right]
                        if pd.isna(first["exit_date_time"]) or second["entry_date_time"] < first["exit_date_time"]:
                            overlap_rows.append(first["_source_row"])
                            overlap_rows.append(second["_source_row"])
            date_time_rule(source, table_name, df, matches_values(df['_source_row'], overlap_rows), 'overlapping_unit_stays')

    # link radiology through its order
    orders = candidates["diagnostic_order"][["order_id", "admission_id", "request_date_time"]]
    compared = pd.merge(candidates["radiology_examination"], orders, on="order_id", how="left")
    compared = pd.merge(compared, parent_times, on="admission_id", how="left")
    for parent_field in ["request_date_time", "admission_date_time", "_parent_discharge"]:
        bad = compared["examination_date_time"] < compared[parent_field]
        reason = "examination_before:" + parent_field
        if parent_field == "_parent_discharge":
            bad = compared["examination_date_time"] > compared[parent_field]
            reason = "examination_after_discharge"
        date_time_rule(source, 'radiology_examination', compared, bad, reason)

# check child rows again after removing admissions or orders
check_relationships()


# Cleaning summary


In [21]:
# collect quarantined rows by source and table
Quarantine_Dict = {}
for source, tables in Original_Dict.items():
    Quarantine_Dict[source] = {}
    for table_name, df in tables.items():
        empty = df.head(0).copy()
        empty["_reason"] = ""
        empty["_missing_required_fields"] = ""
        Quarantine_Dict[source][table_name] = empty
for part in Quarantine_Parts:
    source = part["_source"].iloc[0]
    table_name = part["_table"].iloc[0]
    previous = Quarantine_Dict[source][table_name]
    Quarantine_Dict[source][table_name] = pd.concat([previous, part], ignore_index=True)

summary_rows = []
for source, tables in Cleaned_Dict.items():
    for table_name, df in tables.items():
        raw = Original_Dict[source][table_name]
        rejected = Quarantine_Dict[source][table_name]
        key = Primary_Keys[table_name]

        # check that no rows were lost and ids are unique
        if len(raw) != len(df) + len(rejected):
            raise ValueError(f"row counts do not match for {source}, {table_name}")
        if len(df.drop_duplicates(subset=[key])) != len(df):
            raise ValueError(f"duplicate ids in {source}, {table_name}")

        summary_rows.append({
            "source": source, "table": table_name, "before": len(raw),
            "retained": len(df), "quarantined": len(rejected),
        })

Cleaning_Summary = pd.DataFrame(summary_rows)
print(Cleaning_Summary)


   source                  table  before  retained  quarantined
0       A                patient     800       760           40
1       A              admission    1200      1100          100
2       A          clinical_note    1113       988          125
3       A              diagnosis    2119      1865          254
4       A       diagnostic_order     275       236           39
5       A       discharge_record    1176      1020          156
6       A              icd9_dict      20        20            0
7       A               icu_stay     355       158          197
8       A  radiology_examination     275       234           41
9       A       radiology_report     260         0          260
10      A     surgical_procedure     247       166           81
11      A      triage_assessment    1200      1047          153
12      A              physician      15        15            0
13      A                  nurse      10        10            0
14      B              admission    1200

# Examples


In [22]:
# compare a few retained rows before and after cleaning
examples = [
    ("patient", "gender", "gender"),
    ("admission", "admission_date_time", "admission_date_time"),
    ("triage_assessment", "temp_fahrenheit", "body_temperature"),
]

for table_name, before_field, after_field in examples:
    cleaned = Cleaned_Dict["B"][table_name].head(3)
    raw = Original_Dict["B"][table_name]
    print(f"\nb {table_name}")
    print("before:")
    print(raw.loc[cleaned.index, ["_source_row", before_field]])
    print("after:")
    print(cleaned[["_source_row", after_field]])



b patient
before:
   _source_row gender
0            1      M
1            2      M
2            3      M
after:
   _source_row gender
0            1   MALE
1            2   MALE
2            3   MALE

b admission
before:
   _source_row admission_date_time
0            1    15/04/2026 06:00
1            2    11/06/2026 19:00
2            3    14/05/2026 07:00
after:
   _source_row admission_date_time
0            1 2026-04-15 06:00:00
1            2 2026-06-11 19:00:00
2            3 2026-05-14 07:00:00

b triage_assessment
before:
   _source_row temp_fahrenheit
0            1            99.7
1            2            97.7
2            3           100.0
after:
   _source_row  body_temperature
0            1         37.611111
1            2              36.5
2            3         37.777778


# Save files


In [23]:
from os.path import isdir
from shutil import rmtree

# clear old output folders
def clear_outputs():
    for folder in [clean_output, quarantine_output]:
        if isdir(folder):
            rmtree(folder)
        makedirs(folder, exist_ok=True)


# format dates before exporting
def export_clean(source, df, path):
    exported = df.copy()
    for field in Date_Fields:
        if field in exported:
            date_format = "%Y-%m-%dT%H:%M:%S"
            if field in Date_Only_Fields:
                date_format = "%Y-%m-%d"
            elif source == "D":
                date_format = date_format + "Z"
            exported[field] = exported[field].dt.strftime(date_format)
    exported.to_csv(path, index=False, na_rep="")


clear_outputs()

# save each table in its source folder
for source, tables in Cleaned_Dict.items():
    source_clean_output = f"{clean_output}/Source{source}"
    source_quarantine_output = f"{quarantine_output}/Source{source}"
    for folder in [source_clean_output, source_quarantine_output]:
        makedirs(folder, exist_ok=True)
    for table_name, df in tables.items():
        filename = f"{table_name}.csv"
        export_clean(source, df, f"{source_clean_output}/{filename}")
        Quarantine_Dict[source][table_name].to_csv(f"{source_quarantine_output}/{filename}", index=False, na_rep="")

print("saved to", clean_output, "and", quarantine_output)


saved to ./Cleaned_Source and ./Quarantined_Source


# Quarantine summary


In [24]:
# totals for each source
print(Cleaning_Summary.groupby("source")[["before", "retained", "quarantined"]].sum())

# count the reasons rows were removed
for source, tables in Quarantine_Dict.items():
    for table_name, df in tables.items():
        if len(df) > 0:
            print(f"\n{source} {table_name}")
            print(df.groupby("_reason")["_source_row"].count())


        before  retained  quarantined
source                               
A         9065      7619         1446
B         8714      6906         1808
C         8799      6203         2596
D         8774      5168         3606

A patient
_reason
birth_date_after_review_date                    8
possible_duplicate_patient:insurance_number    32
Name: _source_row, dtype: int64

A admission
_reason
discharge_before_admission               24
missing_or_quarantined_parent:patient    76
Name: _source_row, dtype: int64

A clinical_note
_reason
creation_date:after_discharge               24
missing_or_quarantined_parent:admission    101
Name: _source_row, dtype: int64

A diagnosis
_reason
both_primary_and_secondary_diagnosis        41
diagnosis_date_time:after_discharge         40
missing_or_quarantined_parent:admission    137
patient_does_not_match_admission            36
Name: _source_row, dtype: int64

A diagnostic_order
_reason
missing_or_quarantined_parent:admission    18
patient_does_n